In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, load_npz, csr_matrix
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from tqdm import tqdm
import pickle
import gzip
import re
import os
import numpy as np

In [ ]:
def readGz(path):
    for l in gzip.open(path, 'rt'):
        yield eval(l)

def readCSV(path):
    f = gzip.open(path, 'rt')
    f.readline()
    for l in f:
        yield l.strip().split(',')

def clean(t):
    t = t.lower()
    t = re.sub(r"[^a-z0-9\s']", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def extract_text_features(raw_texts):
    """Extract text features: exclamation points, question marks, periods, 
    text length, newlines, spaces, uppercase letters."""
    n = len(raw_texts)
    features = np.zeros((n, 7))
    
    for i in tqdm(range(n), desc="Extracting text features"):
        raw_text = raw_texts[i]
        features[i, 0] = raw_text.count('!')           # exclamation points
        features[i, 1] = raw_text.count('?')           # question marks
        features[i, 2] = raw_text.count('.')          # periods
        features[i, 3] = len(raw_text)                # text length (chars)
        features[i, 4] = raw_text.count('\n')          # newlines
        features[i, 5] = raw_text.count(' ')            # spaces
        features[i, 6] = sum(1 for c in raw_text if c.isupper())  # uppercase letters
    
    return features

# Load only 10k data points for initial comparison
print("Loading 10k data points...")
data_10k = []
for i, d in enumerate(tqdm(readGz("train_Category.json.gz"), desc="Loading data")):
    if i >= 10000:
        break
    data_10k.append(d)

raw_texts_10k = [d["review_text"] for d in data_10k]  # Keep raw texts for feature extraction
texts_10k = [clean(d["review_text"]) for d in data_10k]
y_10k = np.array([d["genreID"] for d in data_10k])
ratings_10k = np.array([d.get("rating", 0.0) for d in data_10k], dtype=float)
print(f"Loaded {len(data_10k)} data points")

Loading 10k data points...


Loading data: 10000it [00:01, 8640.23it/s]


Loaded 10000 data points


In [ ]:
Xw_file = "Xw_features.npz"
Xc_file = "Xc_features.npz"
word_file = "vectorize_words.pkl"
char_file = "vectorize_chars.pkl"
    
def load_text_features(texts, read=False):
    """Load or compute TF-IDF features. If read=True, loads from cache (full dataset).
    If read=False, computes on provided texts."""
    if read:
        print("Loading saved vectorized features (full dataset)...")
        Xw = load_npz(Xw_file)
        Xc = load_npz(Xc_file)
        with open(word_file, 'rb') as f:
            word_vec = pickle.load(f)
        with open(char_file, 'rb') as f:
            char_vec = pickle.load(f)
        print(f"Loaded Xw shape: {Xw.shape}, Xc shape: {Xc.shape}")
    else:
        print("Computing TF-IDF features on provided data...")
        word_vec = TfidfVectorizer(
            max_features=20000,
            ngram_range=(1,3),
            min_df=2,
            stop_words="english"
        )
        char_vec = TfidfVectorizer(
            analyzer="char",
            ngram_range=(3,5),
            min_df=2,
            max_features=30000
        )
        Xw = word_vec.fit_transform(texts)
        Xc = char_vec.fit_transform(texts)
        print(f"Computed Xw shape: {Xw.shape}, Xc shape: {Xc.shape}")
    return Xw, Xc, word_vec, char_vec

In [ ]:
# Compute features on 10k data only
print("\n" + "="*60)
print("Feature Extraction on 10k Data Points")
print("="*60)

Xw_10k, Xc_10k, word_vec_10k, char_vec_10k = load_text_features(texts_10k, read=False)

# Extract text features from raw texts (10k only)
print("\nExtracting text features from 10k data...")
X_text_features_10k = extract_text_features(raw_texts_10k)

# Append rating as an additional feature
ratings_10k_column = ratings_10k.reshape(-1, 1)
X_text_features_with_rating_10k = np.hstack([X_text_features_10k, ratings_10k_column])

# Normalize text features (they have very different scales!)
print("Normalizing text + rating features...")
scaler = StandardScaler()
X_text_features_10k_normalized = scaler.fit_transform(X_text_features_with_rating_10k)

# Convert to CSR format for efficient sparse operations
X_text_features_sparse_10k = csr_matrix(X_text_features_10k_normalized)
print(f"Text + rating features shape: {X_text_features_with_rating_10k.shape} (normalized)")

# Ensure TF-IDF features are in CSR format
Xw_10k = Xw_10k.tocsr() if hasattr(Xw_10k, 'tocsr') else Xw_10k
Xc_10k = Xc_10k.tocsr() if hasattr(Xc_10k, 'tocsr') else Xc_10k

# Combine TF-IDF features with text features (10k only)
X_baseline_10k = hstack([Xw_10k, Xc_10k]).tocsr()  # Baseline: only TF-IDF, ensure CSR
X_enhanced_10k = hstack([Xw_10k, Xc_10k, X_text_features_sparse_10k]).tocsr()  # Enhanced: TF-IDF + text features, ensure CSR

print(f"\n10k Baseline features shape: {X_baseline_10k.shape}")
print(f"10k Enhanced features shape: {X_enhanced_10k.shape}")


Feature Extraction on 10k Data Points
Computing TF-IDF features on provided data...
Computed Xw shape: (10000, 20000), Xc shape: (10000, 30000)

Extracting text features from 10k data...


Extracting text features: 100%|██████████| 10000/10000 [00:00<00:00, 18217.85it/s]


Normalizing text + rating features...
Text + rating features shape: (10000, 8) (normalized)

10k Baseline features shape: (10000, 50000)
10k Enhanced features shape: (10000, 50008)


In [ ]:
# Load baseline 5-CV results for reference
baseline_5cv_file = "baseline_5cv_results.pkl"
if os.path.exists(baseline_5cv_file):
    print("Loading baseline 5-CV results (for reference)...")
    with open(baseline_5cv_file, 'rb') as f:
        baseline_5cv = pickle.load(f)
    print(f"Saved baseline 5-CV scores (full dataset): {baseline_5cv['cv_scores']}")
    print(f"Saved baseline 5-CV mean (full dataset): {baseline_5cv['mean']:.4f} (+/- {baseline_5cv['std']*2:.4f})")
else:
    print("Warning: baseline_5cv_results.pkl not found.")
    baseline_5cv = None

# Perform 5-fold cross-validation on 10k data
print("\n" + "="*60)
print("5-Fold Cross-Validation Comparison on 10k Data")
print("="*60)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Baseline model (TF-IDF only) on 10k
# Use dual=False for better performance when n_samples < n_features
print("\nTraining baseline model (TF-IDF only) with 5-CV on 10k data...")
baseline_scores_10k = []
fold_splits = list(kfold.split(X_baseline_10k))
for train_idx, val_idx in tqdm(fold_splits, total=5, desc="Baseline CV (10k)"):
    mod_baseline = LinearSVC(C=0.1, random_state=42, max_iter=10000, dual=False)
    mod_baseline.fit(X_baseline_10k[train_idx], y_10k[train_idx])
    score = mod_baseline.score(X_baseline_10k[val_idx], y_10k[val_idx])
    baseline_scores_10k.append(score)

baseline_mean_10k = np.mean(baseline_scores_10k)
baseline_std_10k = np.std(baseline_scores_10k)
print(f"Baseline CV Scores (10k): {baseline_scores_10k}")
print(f"Baseline Mean (10k): {baseline_mean_10k:.4f} (+/- {baseline_std_10k*2:.4f})")

# Enhanced model (TF-IDF + text features) on 10k
# Use dual=False for better performance when n_samples < n_features
print("\nTraining enhanced model (TF-IDF + text features) with 5-CV on 10k data...")
enhanced_scores_10k = []
for train_idx, val_idx in tqdm(fold_splits, total=5, desc="Enhanced CV (10k)"):
    mod_enhanced = LinearSVC(C=0.1, random_state=42, max_iter=10000, dual=False)
    mod_enhanced.fit(X_enhanced_10k[train_idx], y_10k[train_idx])
    score = mod_enhanced.score(X_enhanced_10k[val_idx], y_10k[val_idx])
    enhanced_scores_10k.append(score)

enhanced_mean_10k = np.mean(enhanced_scores_10k)
enhanced_std_10k = np.std(enhanced_scores_10k)
print(f"Enhanced CV Scores (10k): {enhanced_scores_10k}")
print(f"Enhanced Mean (10k): {enhanced_mean_10k:.4f} (+/- {enhanced_std_10k*2:.4f})")

# Comparison on 10k
print("\n" + "="*60)
print("Comparison Results (10k Data)")
print("="*60)
print(f"Baseline (TF-IDF only):     {baseline_mean_10k:.4f} (+/- {baseline_std_10k*2:.4f})")
print(f"Enhanced (TF-IDF + features): {enhanced_mean_10k:.4f} (+/- {enhanced_std_10k*2:.4f})")
improvement_10k = enhanced_mean_10k - baseline_mean_10k
relative_improvement = improvement_10k / baseline_mean_10k * 100
print(f"Improvement: {improvement_10k:.4f} ({relative_improvement:.2f}%)")

# Check if enhanced model is within 2% of baseline or better
threshold = -0.02  # Enhanced should be within 2% (i.e., at most 2% worse)
performance_diff = (enhanced_mean_10k - baseline_mean_10k) / baseline_mean_10k

print(f"\nPerformance difference: {performance_diff:.4f} ({performance_diff*100:.2f}%)")
print(f"Threshold: {threshold:.4f} ({threshold*100:.2f}%)")

if performance_diff < threshold:
    error_msg = (
        f"Enhanced model performance ({enhanced_mean_10k:.4f}) is more than 2% worse than baseline "
        f"({baseline_mean_10k:.4f}). Performance difference: {performance_diff*100:.2f}%. "
        f"Not training on full dataset."
    )
    raise Exception(error_msg)

print(f"\n✓ Enhanced model is within 2% of baseline or better!")
print(f"  Proceeding to load full dataset and train models...")

# Load full dataset and train
print("\n" + "="*60)
print("Loading Full Dataset and Training Models")
print("="*60)

# Load full dataset
print("Loading full dataset...")
data_full = [d for d in tqdm(readGz("train_Category.json.gz"), desc="Loading full data")]
raw_texts_full = [d["review_text"] for d in data_full]
texts_full = [clean(d["review_text"]) for d in tqdm(data_full, desc="Cleaning texts")]
y_full = np.array([d["genreID"] for d in data_full])
ratings_full = np.array([d.get("rating", 0.0) for d in data_full], dtype=float)
n_total = len(data_full)
print(f"Loaded {n_total} data points")

# Compute features on full dataset
print("\nComputing TF-IDF features on full dataset...")
Xw_full, Xc_full, word_vec_full, char_vec_full = load_text_features(texts_full, read=False)

print("\nExtracting text features from full dataset...")
X_text_features_full = extract_text_features(raw_texts_full)
ratings_full_column = ratings_full.reshape(-1, 1)
X_text_features_with_rating_full = np.hstack([X_text_features_full, ratings_full_column])

# Normalize using the same scaler (fit on 10k, transform on full)
print("Normalizing text + rating features (using scaler from 10k)...")
X_text_features_full_normalized = scaler.transform(X_text_features_with_rating_full)
X_text_features_sparse_full = csr_matrix(X_text_features_full_normalized)

# Ensure CSR format
Xw_full = Xw_full.tocsr() if hasattr(Xw_full, 'tocsr') else Xw_full
Xc_full = Xc_full.tocsr() if hasattr(Xc_full, 'tocsr') else Xc_full

# Combine features
X_baseline_full = hstack([Xw_full, Xc_full]).tocsr()
X_enhanced_full = hstack([Xw_full, Xc_full, X_text_features_sparse_full]).tocsr()

print(f"\nFull dataset - Baseline features shape: {X_baseline_full.shape}")
print(f"Full dataset - Enhanced features shape: {X_enhanced_full.shape}")

# Train on full dataset (load baseline model, train enhanced model)
baseline_model_file = "baseline_model.pkl"
print(f"\nLoading baseline model from {baseline_model_file} (pretrained on full dataset)...")
if not os.path.exists(baseline_model_file):
    raise FileNotFoundError(f"Baseline model file '{baseline_model_file}' not found. Please provide the pretrained baseline model.")
with open(baseline_model_file, 'rb') as f:
    mod_baseline_full = pickle.load(f)
print("✓ Baseline model loaded from disk.")

print(f"\nTraining enhanced model on full dataset ({n_total} samples)...")
print("This may take a while (no progress bar available for sklearn.fit)...")
mod_enhanced_full = LinearSVC(C=0.1, random_state=42, max_iter=10000, dual=False)
mod_enhanced_full.fit(X_enhanced_full, y_full)
print("✓ Enhanced model trained on full dataset.")

print("\n✓ Both models trained successfully on full dataset!")

if baseline_5cv is not None:
    print(f"\nNote: Saved baseline 5-CV mean (full dataset): {baseline_5cv['mean']:.4f} (+/- {baseline_5cv['std']*2:.4f})")

Loading baseline 5-CV results (for reference)...
Saved baseline 5-CV scores (full dataset): [0.76955 0.76235 0.7662  0.7691  0.7689 ]
Saved baseline 5-CV mean (full dataset): 0.7672 (+/- 0.0054)

5-Fold Cross-Validation Comparison on 10k Data

Training baseline model (TF-IDF only) with 5-CV on 10k data...


Baseline CV (10k): 100%|██████████| 5/5 [00:42<00:00,  8.57s/it]


Baseline CV Scores (10k): [0.6925, 0.6975, 0.708, 0.68, 0.708]
Baseline Mean (10k): 0.6972 (+/- 0.0210)

Training enhanced model (TF-IDF + text features) with 5-CV on 10k data...


Enhanced CV (10k): 100%|██████████| 5/5 [02:15<00:00, 27.11s/it]


Enhanced CV Scores (10k): [0.6915, 0.7025, 0.705, 0.6835, 0.706]
Enhanced Mean (10k): 0.6977 (+/- 0.0176)

Comparison Results (10k Data)
Baseline (TF-IDF only):     0.6972 (+/- 0.0210)
Enhanced (TF-IDF + features): 0.6977 (+/- 0.0176)
Improvement: 0.0005 (0.07%)

Performance difference: 0.0007 (0.07%)
Threshold: -0.0200 (-2.00%)

✓ Enhanced model is within 2% of baseline or better!
  Proceeding to load full dataset and train models...

Loading Full Dataset and Training Models
Loading full dataset...


Loading full data: 100000it [00:08, 12249.59it/s]
Cleaning texts: 100%|██████████| 100000/100000 [00:07<00:00, 12933.31it/s]


Loaded 100000 data points

Computing TF-IDF features on full dataset...
Computing TF-IDF features on provided data...


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001BCFA676DB0>>
Traceback (most recent call last):
  File "c:\Users\issac\miniforge3\envs\dsc80\Lib\site-packages\ipykernel\ipkernel.py", line 790, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\issac\miniforge3\envs\dsc80\Lib\threading.py", line 1535, in enumerate
    def enumerate():
    
KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
# Compare baseline vs enhanced accuracies on the full dataset
print("\n" + "="*60)
print("Accuracy Comparison on Full Dataset")
print("="*60)

baseline_full_acc = mod_baseline_full.score(X_baseline_full, y_full)
enhanced_full_acc = mod_enhanced_full.score(X_enhanced_full, y_full)
full_improvement = enhanced_full_acc - baseline_full_acc
full_relative_improvement = full_improvement / baseline_full_acc * 100 if baseline_full_acc != 0 else 0

print(f"Baseline accuracy (full data):   {baseline_full_acc:.4f}")
print(f"Enhanced accuracy (full data):   {enhanced_full_acc:.4f}")
print(f"Absolute improvement:            {full_improvement:.4f}")
print(f"Relative improvement:            {full_relative_improvement:.2f}%")

if full_improvement > 0:
    print("Enhanced model outperforms baseline on the full dataset.")
elif full_improvement < 0:
    print("Enhanced model underperforms baseline on the full dataset.")
else:
    print("Enhanced model matches baseline accuracy on the full dataset.")



Accuracy Comparison on Full Dataset
Baseline accuracy (full data):   0.8600
Enhanced accuracy (full data):   0.8600
Absolute improvement:            0.0000
Relative improvement:            0.00%
Enhanced model outperforms baseline on the full dataset.


In [ ]:
# Write enhanced model predictions to output file
output_file = "predictions_Category.csv"
print(f"\nWriting enhanced model predictions to {output_file} ...")

with open(output_file, "w") as predictions:
    predictions.write("userID,reviewID,prediction\n")
    for entry in tqdm(readGz("test_Category.json.gz"), desc="Predicting (enhanced)"):
        raw_text = entry["review_text"]
        text = clean(raw_text)

        # TF-IDF features using full-data vectorizers
        Xw_test = word_vec_full.transform([text])
        Xc_test = char_vec_full.transform([text])

        # Text features (same 7 features as training) + rating
        rating = entry.get('rating', 0.0)
        text_feat = np.array([[
            raw_text.count('!'),
            raw_text.count('?'),
            raw_text.count('.'),
            len(raw_text),
            raw_text.count('\n'),
            raw_text.count(' '),
            sum(1 for c in raw_text if c.isupper()),
            rating,
        ]])
        text_feat_norm = scaler.transform(text_feat)
        X_text_feat = csr_matrix(text_feat_norm)

        # Combine all features
        X_test = hstack([Xw_test, Xc_test, X_text_feat])

        pred = mod_enhanced_full.predict(X_test)[0]
        predictions.write(f"{entry['user_id']},{entry['review_id']},{pred}\n")

print(f"✓ Predictions written to {output_file}")



Writing enhanced model predictions to predictions_Category.csv ...


Predicting (enhanced): 10000it [01:09, 144.28it/s]

✓ Predictions written to predictions_Category.csv


In [ ]:
# model_file = "baseline_model.pkl"
# if os.path.exists(model_file):
#     print("Loading saved baseline model...")
#     with open(model_file, 'rb') as f:
#         mod = pickle.load(f)
#     print(f"Model loaded. Validation score: {mod.score(Xvalid, yvalid)}")
# else:
#     print("Training baseline model (this may take a while)...")
#     mod = LinearSVC(C=0.1)
#     mod.fit(Xtrain, ytrain)
#     score = mod.score(Xvalid, yvalid)
#     print(f"Model trained. Validation score: {score}")
#     with open(model_file, 'wb') as f:
#         pickle.dump(mod, f)
#     print("Model saved.")

In [ ]:
# predictions = open("predictions_Category.csv", "w")
# predictions.write("userID,reviewID,prediction\n")

# for l in tqdm(readGz("test_Category.json.gz"), desc="Writing category predictions"):
#     text = clean(l["review_text"])

#     Xw = word_vec.transform([text])
#     Xc = char_vec.transform([text])
#     Xtest = hstack([Xw, Xc])

#     pred = mod.predict(Xtest)[0]
#     predictions.write(f"{l['user_id']},{l['review_id']},{pred}\n")

# predictions.close()